In [275]:
import re
from collections import Counter
from typing import List, Dict, Tuple
from minitorch.tensor.tensor import Tensor

In [62]:
class Tokenizer:
    """
    Base Tokenizer providing the basic architecture that tokenizers follow.
    
    It provides the following architecture
    - encode(): convert text to token ID.
    - decode(): convert token IDs back to text.
    """
    
    #* predefined tokens
    TOKEN_UNKNOWN= "<UNK>"
    TOKEN_EOT = "<EOT>"
    
    def encode(self, text: str) -> List[int]:
        """
        Convert text to token IDs.
        
        Args:
            text (str): The input text to be tokenized.
        
        Returns:
            List[int]: A list of token IDs corresponding to the input text.
        """
        raise NotImplementedError(
            f"encode() not implemented on Tokenizer class.\n"
            f" ❌ Called encode() on abstract base class {self.__class__.__name__}\n"
            f" 🔦 Tokenizer class is just an interface. Implement encode() in concrete class such as CharTokenizer() or BPETokennizer().\n"
            f" ✅ Example: tokenizer = CharTokenizer(); token_ids = tokenizer.encode('hello world')")
        
    def decode(self, token_ids: List[int]) -> str:
        """
        Convert token ids back to raw text

        Args:
            token_ids (List[int]): IDs to convert to text

        Returns:
            str: A string of text obtained from the token Ids
        """
        raise NotImplementedError(
            f"decode() not implemented on Tokenizer class.\n"
            f" ❌ Called decode() on abstract base class {self.__class__.__name__}\n"
            f" 🔦 Tokenizer class is just an interface. Implement decode() in concrete class such as CharTokenizer() or BPETokennizer().\n"
            f" ✅ Example: tokenizer = CharTokenizer(); text = tokenizer.decode([1, 2, 3])")
    

class CharTokenizer(Tokenizer):
    """
    Character-level tokenizer that converts text into token IDs based on individual characters.
    
    This tokenizer creates a vocabulary of unique characters from the input text and assigns a unique ID to each character. 
    It also includes special tokens for unknown characters and end-of-text.
    """
    
    def __init__(self):
        self.char_to_id: Dict[str, int] = {}
        self.id_to_char: Dict[int, str] = {}
        self.next_id: int = 0
        
        # Add special tokens to the vocabulary
        self._add_token(self.TOKEN_UNKNOWN)
        self._add_token(self.TOKEN_EOT)
        
    def _add_token(self, token: str) -> None:
        """
        Add a token to the vocabulary and return its assigned ID.
        
        Args:
            token (str): The token to be added to the vocabulary.
        
        Returns:
            None
        """
        if token not in self.char_to_id:
            self.char_to_id[token] = self.next_id
            self.id_to_char[self.next_id] = token
            self.next_id += 1
            
        self.vocab_size : int = len(self.char_to_id)
    
    def build_vocab(self, corpus: List[str]) -> None:
        """
        Build the vocabulary from a list of text samples.
        
        Args:
            corpus (List[str]): A list of text samples to build the vocabulary from.
        """
        #* lower case and strip whitespace from each text sample
        corpus = [text.lower().strip() for text in corpus]
        
        #* Iterate through each text sample and add each character to the vocabulary
        for text in corpus:
            for char in text:
                self._add_token(char)
                
    def encode(self, text: str) -> List[int]:
        """
        Convert text to token IDs based on the character-level vocabulary.
        
        Args:
            text (str): The input text to be tokenized.
        
        Returns:
            List[int]: A list of token IDs corresponding to the input text.
        """
        token_ids = []
        for char in text:
            token_id = self.char_to_id.get(char, self.char_to_id[self.TOKEN_UNKNOWN])
            token_ids.append(token_id)
        return token_ids
    
    def decode(self, token_ids: List[int]) -> str:
        """
        Convert token IDs back to raw text based on the character-level vocabulary.
        
        Args:
            token_ids (List[int]): A list of token IDs to be converted back to text.
        
        Returns:
            str: A string of text obtained from the token IDs.
        """
        chars = []
        for token_id in token_ids:
            char = self.id_to_char.get(token_id, 0)
            chars.append(char)
        return ''.join(chars)

In [63]:
tokenizer = CharTokenizer()
tokenizer.build_vocab(["hello world",
                    "hi there", 
                    "how are you?",
                    "my name is Boniface Mwangangi and short and small than you",
                    "i am a student at the university of nairobi as it's the best school in the country",
                    "I have a bachelor's degree in computer science from the university of nairobi",
                    "I was born in Meru county in Kenya, but lived most of my life in Nanyuki in Laikipia county.",
                    "Mourine lives in Makima market and I love her, but never again after today",
                    "Girlfriend is a term used to describe a romantic partner who is female"
                    ])

In [52]:
token_ids = tokenizer.encode("meru as a county is smaller than laikipia county")
raw_text = tokenizer.decode(token_ids)

In [64]:
tokenizer.vocab_size

29

In [353]:
class BPETokenizer(Tokenizer):
    def __init__(self, vocab_size: int = 10000):
        self.vocab_size:     int = vocab_size
        self.vocab:          List = []
        self.tokens_to_ids:  Dict[str, int] = {}
        self.ids_to_tokens:  Dict[int, str] = {}
        self.merges:         List[Tuple[str, str]] = []
        
    def pre_tokenize(self, text: str) -> List[str]:
        """
        Splits text into words and punctuation while preserving 
        essential boundaries.
        """
        # This regex separates words from punctuation:
        # \w+ matches word characters
        # [^\w\s] matches punctuation (anything not word/space)
        return re.findall(r'\w+|[^\w\s]', text)
        
    def _get_word_tokens(self, word: str) -> List[str]:
        """
        Get the individual tokens of a word

        Args:
            word (str): The word to get tokens from

        Returns:
            List[str]: Individual tokens from a word
        """
        tokens = list(word)
        tokens[-1] += Tokenizer.TOKEN_EOT
        return tokens
    
    
    def train(self, corpus: List[str], vocab_size: int):
        """
        Train the BPE to learn pairs and merge them.
        
        It initializes character vocabulary and run a greedy merge loop
        using _count_byte_pairs to find the best pair and _merge_pairs()
        to merge them

        Args:
            corpus (List[str]): Document to learn and train from
            vocab_size (int): maximum vocabulary size allowed
        """
        full_corpus = []
        
        for sentence in corpus:
            full_corpus.extend(self.pre_tokenize(sentence))

        if vocab_size:
            self.vocab_size = vocab_size
            
            
        word_freq: Counter = Counter(full_corpus)
        word_tokens: Dict[str, list[str]] = {}
        vocab = set()
        
        #* get the words tokens from corpus
        for word in full_corpus:
            tokens = self._get_word_tokens(word)
            word_tokens[word] = tokens
            vocab.update(tokens)
        
        #* update the vocabulary using the tokens and
        #* unknown token    
        self.vocab = sorted(list(vocab))
        
        if Tokenizer.TOKEN_UNKNOWN not in self.vocab:
            self.vocab.insert(0, Tokenizer.TOKEN_UNKNOWN)
            
        #* find the best pair(s) and merge them
        while len(self.vocab) < self.vocab_size:
            pair_counts = self._count_byte_pairs(word_tokens, word_freq)
            if not pair_counts:
                break
            
            best_pair = pair_counts.most_common(1)[0][0]
            new_token = self._merge_pair(word_tokens, best_pair)
    
            self.merges.append(best_pair)
            self.vocab.append(new_token)
            
        self._build_mapping()
        
    def _build_mapping(self):
        """
        Create token to id and id to token mappings
        """
        self.tokens_to_ids = {token: id for id, token in enumerate(self.vocab)}
        self.ids_to_tokens = {id: token for id, token in enumerate(self.vocab)}
            
    def _count_byte_pairs(self, word_token: Dict[str, List[str]], word_count: Counter)-> Counter:
        """
        Count the frequency of all adjacent token pairs in a corpus
        
        Each pair count is weighted by the frequency of word containing it
        in the corpus, so most frequent words contribute more the statistic

        Args:
            word_token (Dict[str, List[str]]): Dictionary with word and its tokens
            word_count (Counter): Dictionary showing the frequency count of the word

        Returns:
            Counter: Pair count freqeuncies of the adjacent tokens
        """
        pair_count = Counter()
        for word, count in word_count.items():
            tokens = word_token[word]
            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                pair_count[pair] += count
                
        return pair_count
                
    def _merge_pair(self, word_token: Dict[str, List[str]], pair: Tuple[str, str]) -> str:
        """
        Merge the most frequent pair in all word token lists.

        Scans through every word's tokens and replaces adjacent occurrences
        of the pair with a single concatenated token. Modifies word_tokens
        in place and returns the new merged token string.
        """
        merged_pair = pair[0] + pair[1]
        
        for word in word_token:
            tokens = word_token[word]
            new_tokens: List[str] = []
            counter = 0
            
            
            while counter < len(tokens):
                if (counter < len(tokens) - 1) and tokens[counter] == pair[0]\
                    and tokens[counter + 1] == pair[1]:
                    new_tokens.append(merged_pair)
                    counter += 2
                else:
                    new_tokens.append(tokens[counter])
                    counter += 1
            word_token[word] = new_tokens
                    
        return merged_pair
    
    def _apply_merges(self, tokens: List[str]) -> List[str]:
        if not self.merges:
            return tokens
        
        current_tokens = list(tokens)
        for pair in self.merges:
            new_tokens = []
            i = 0
            
            while (i < len(current_tokens)):
                if (i < len(current_tokens) - 1) and\
                current_tokens[i] == pair[0] and \
                current_tokens[i + 1] == pair[1]:
                    new_tokens.append(pair[0] + pair[1])
                    
                    i += 2
                else:
                    new_tokens.append(current_tokens[i])
                    i += 1
                    
            current_tokens = new_tokens
        return current_tokens
    
    def encode(self, text: str)-> List[int]:
        words = text.split()
        all_tokens = []
        
        for word in words:
            tokens = self._get_word_tokens(word)
            merged_tokens = self._apply_merges(tokens)
            all_tokens.extend(merged_tokens)
        
        tokens_ids = []
        for token in all_tokens:
            id = self.tokens_to_ids.get(token, 0)
            tokens_ids.append(id)
        
        return tokens_ids
    
    def decode(self, token_ids: List[int]) -> str:
        #* return empty string id token mapping doesn't exist
        if not self.ids_to_tokens:
            return ""
        
        #* iterate through the token ids and get the corresponding
        #* token from id to token mapping
        text = [self.ids_to_tokens.get(t, Tokenizer.TOKEN_UNKNOWN) for t in token_ids]
        
        #* join all the tokens together and perform some clean up
        text = "".join(text)
        text = text.replace(Tokenizer.TOKEN_EOT, " ")
        text = " ".join(text.split())
        return text
        
            

In [356]:
corpus = ["hello, world!.",
            "hi there.", 
            "how are you?",
            "my name is Boniface Mwangangi and short and small than you.",
            "I am a student at the university of nairobi as it's the best school in the country.",
            "I have a bachelor's degree in computer science from the university of nairobi.",
            "I was born in Meru county in Kenya, but lived most of my life in Nanyuki in Laikipia county.",
            "Mourine lives in Makima market and I love her, but never again after today.",
            "Girlfriend is a term used to describe a romantic partner who, is female.",
            "from now on, am leaving and get away from her."
            ]

sentence = "Meru county is smaller than Laikipia county."
tokenizer = BPETokenizer()
tokenizer.train(corpus, vocab_size=100)
token_ids = tokenizer.encode(sentence)
tokenizer.decode(token_ids)

'Meru county is smaller than Laikipia county.'

In [345]:
Counter(all_tokens)

Counter({'a': 34,
         'i': 33,
         'e': 29,
         'o': 29,
         'n': 26,
         'r': 23,
         't': 19,
         'h': 16,
         'e<EOT>': 16,
         'l': 12,
         'u': 12,
         's': 11,
         'n<EOT>': 11,
         'm': 10,
         'c': 10,
         't<EOT>': 10,
         '.<EOT>': 9,
         'y<EOT>': 9,
         'd<EOT>': 8,
         's<EOT>': 8,
         'f': 8,
         'v': 8,
         'b': 8,
         'a<EOT>': 7,
         'r<EOT>': 7,
         'm<EOT>': 6,
         ',<EOT>': 5,
         'w': 5,
         'i<EOT>': 5,
         'g': 5,
         'y': 4,
         'M': 4,
         'I<EOT>': 4,
         'd': 4,
         'k': 4,
         'o<EOT>': 3,
         'u<EOT>': 3,
         'f<EOT>': 3,
         'p': 3,
         'w<EOT>': 2,
         'l<EOT>': 2,
         "'<EOT>": 2,
         '!<EOT>': 1,
         '?<EOT>': 1,
         'B': 1,
         'K': 1,
         'N': 1,
         'L': 1,
         'G': 1,
         'c<EOT>': 1,
         'g<EOT>': 1})

In [338]:
for word in full_corpus:
    all_tokens.extend(get_word_tokens(word))

In [339]:
all_tokens

['h',
 'e',
 'l',
 'l',
 'o<EOT>',
 ',<EOT>',
 'w',
 'o',
 'r',
 'l',
 'd<EOT>',
 '!<EOT>',
 '.<EOT>',
 'h',
 'i<EOT>',
 't',
 'h',
 'e',
 'r',
 'e<EOT>',
 '.<EOT>',
 'h',
 'o',
 'w<EOT>',
 'a',
 'r',
 'e<EOT>',
 'y',
 'o',
 'u<EOT>',
 '?<EOT>',
 'm',
 'y<EOT>',
 'n',
 'a',
 'm',
 'e<EOT>',
 'i',
 's<EOT>',
 'B',
 'o',
 'n',
 'i',
 'f',
 'a',
 'c',
 'e<EOT>',
 'M',
 'w',
 'a',
 'n',
 'g',
 'a',
 'n',
 'g',
 'i<EOT>',
 'a',
 'n',
 'd<EOT>',
 's',
 'h',
 'o',
 'r',
 't<EOT>',
 'a',
 'n',
 'd<EOT>',
 's',
 'm',
 'a',
 'l',
 'l<EOT>',
 't',
 'h',
 'a',
 'n<EOT>',
 'y',
 'o',
 'u<EOT>',
 '.<EOT>',
 'I<EOT>',
 'a',
 'm<EOT>',
 'a<EOT>',
 's',
 't',
 'u',
 'd',
 'e',
 'n',
 't<EOT>',
 'a',
 't<EOT>',
 't',
 'h',
 'e<EOT>',
 'u',
 'n',
 'i',
 'v',
 'e',
 'r',
 's',
 'i',
 't',
 'y<EOT>',
 'o',
 'f<EOT>',
 'n',
 'a',
 'i',
 'r',
 'o',
 'b',
 'i<EOT>',
 'a',
 's<EOT>',
 'i',
 't<EOT>',
 "'<EOT>",
 's<EOT>',
 't',
 'h',
 'e<EOT>',
 'b',
 'e',
 's',
 't<EOT>',
 's',
 'c',
 'h',
 'o',
 'o',
 'l<EOT

In [281]:
def pre_tokenize(text: str) -> List[str]:
        """
        Splits text into words and punctuation while preserving 
        essential boundaries.
        """
        # This regex separates words from punctuation:
        # \w+ matches word characters
        # [^\w\s] matches punctuation (anything not word/space)
        return re.findall(r'\w+|[^\w\s]', text)
def get_word_tokens(word: str) -> List[str]:
        """
        Get the individual tokens of a word

        Args:
            word (str): The word to get tokens from

        Returns:
            List[str]: Individual tokens from a word
        """
        tokens = list(word)
        tokens[-1] += Tokenizer.TOKEN_EOT
        return tokens
    
tokens = []
for word in pre_tokenize(corpus[0]):
    token = get_word_tokens(word)
    tokens.append(token)

In [306]:
get_word_tokens(pre_tokenize(corpus[0]))

['hello', ',', 'world', '!', '.<EOT>']

In [308]:
full_corpus = []
for sentence in corpus:
    full_corpus.extend(pre_tokenize(sentence))

In [320]:
all_tokens = []
for word in full_corpus:
    all_tokens.extend(get_word_tokens(word))
    # break

In [321]:
all_tokens


['h',
 'e',
 'l',
 'l',
 'o<EOT>',
 ',<EOT>',
 'w',
 'o',
 'r',
 'l',
 'd<EOT>',
 '!<EOT>',
 '.<EOT>',
 'h',
 'i<EOT>',
 't',
 'h',
 'e',
 'r',
 'e<EOT>',
 '.<EOT>',
 'h',
 'o',
 'w<EOT>',
 'a',
 'r',
 'e<EOT>',
 'y',
 'o',
 'u<EOT>',
 '?<EOT>',
 'm',
 'y<EOT>',
 'n',
 'a',
 'm',
 'e<EOT>',
 'i',
 's<EOT>',
 'B',
 'o',
 'n',
 'i',
 'f',
 'a',
 'c',
 'e<EOT>',
 'M',
 'w',
 'a',
 'n',
 'g',
 'a',
 'n',
 'g',
 'i<EOT>',
 'a',
 'n',
 'd<EOT>',
 's',
 'h',
 'o',
 'r',
 't<EOT>',
 'a',
 'n',
 'd<EOT>',
 's',
 'm',
 'a',
 'l',
 'l<EOT>',
 't',
 'h',
 'a',
 'n<EOT>',
 'y',
 'o',
 'u<EOT>',
 '.<EOT>',
 'I<EOT>',
 'a',
 'm<EOT>',
 'a<EOT>',
 's',
 't',
 'u',
 'd',
 'e',
 'n',
 't<EOT>',
 'a',
 't<EOT>',
 't',
 'h',
 'e<EOT>',
 'u',
 'n',
 'i',
 'v',
 'e',
 'r',
 's',
 'i',
 't',
 'y<EOT>',
 'o',
 'f<EOT>',
 'n',
 'a',
 'i',
 'r',
 'o',
 'b',
 'i<EOT>',
 'a',
 's<EOT>',
 'i',
 't<EOT>',
 "'<EOT>",
 's<EOT>',
 't',
 'h',
 'e<EOT>',
 'b',
 'e',
 's',
 't<EOT>',
 's',
 'c',
 'h',
 'o',
 'o',
 'l<EOT

In [314]:
''.join(full_corpus).split()

["hello,world!.hithere.howareyou?mynameisBonifaceMwangangiandshortandsmallthanyou.Iamastudentattheuniversityofnairobiasit'sthebestschoolinthecountry.Ihaveabachelor'sdegreeincomputersciencefromtheuniversityofnairobi.IwasborninMerucountyinKenya,butlivedmostofmylifeinNanyukiinLaikipiacounty.MourinelivesinMakimamarketandIloveher,butneveragainaftertoday.Girlfriendisatermusedtodescribearomanticpartnerwho,isfemale.fromnowon,amleavingandgetawayfromher."]